# Open-weights rung - reliability probe on Colab (vLLM + Qwen3.6-27B)

Adds an **open-weights, white-box** model to the R90 ladder. Downloads **Qwen/Qwen3.6-27B** (dense, Apache-2.0, native **262,144**-token window), serves it through vLLM's **OpenAI-compatible** endpoint (the probe's OpenAI adapter then works with only an env-var swap), and runs a **near-only smoke test** as the hard go/no-go before any GPU-expensive sweep. **This revision = the high-fill PUSH:** the 8k–212k curve is already banked, so it runs only the **top fills** at **`--workers 1`** (no KV-cache queue — the thing that stalled 172k before), with **`--verbose` live per-run progress** + a **DEBUG cell**, to extend the curve past ctx 212k.

**Footguns this notebook handles for you (all card-verified):**
- **VRAM:** 27B in BF16 is ~54 GB of weights - needs an **80 GB A100**. It will NOT fit a 40 GB A100. Cell 2 checks.
- **No YaRN:** the 262k native window already exceeds every fill we use, so `--rope-scaling` is left **OFF** (the card warns static YaRN degrades shorter texts - which would corrupt the near gate).
- **Thinking OFF:** Qwen3.6 thinks by default and has no `/no_think` switch; we force `enable_thinking=False` per request so it matches the non-reasoning API panel (gpt-3.5 / gpt-4o-mini / Haiku / Sonnet) and the tool call is not starved by a think block.
- **Tool parser:** `--tool-call-parser qwen3_coder` (the probe uses native function-calling).
- **Hang visibility:** the sweep runs with `--verbose` (timestamped line per run) + `--timeout`, and the DEBUG cell dumps KV-cache capacity / GPU memory / recent vLLM log — so a stall is diagnosable, not silent.

In [ ]:
# --- config: PUSH past ctx 212k — top fills only, ONE request at a time (no KV-queue timeout) ---
MODEL = 'Qwen/Qwen3.6-27B'   # dense, Apache-2.0, native 262,144 window
MODEL_DIR = '/content/model'
SERVED = 'open-model'        # same label -> EXTENDS the existing BF16 curve (8k-212k already banked)
MAX_LEN = 262144             # FULL native window, no YaRN
KV_DTYPE = 'auto'            # BF16 KV (clean) — confirmed it fits 262k on the 80 GB A100 last run
GMEM = 0.95
HI_FILLS = '160000 172000'   # NEW points PAST 212k: ctx ~236k / ~253k. (172k timed out at workers=5;
                             # workers=1 below removes the KV-cache queue that caused the timeout.)
WORKERS = 1                  # ONE request at a time -> no KV queue -> no client timeout at huge ctx
TIMEOUT = 360                # per-request fail-fast (s): a legit ~253k prefill is ~1-2 min, so a
                             # stall > 6 min is a real hang -> the harness retries it visibly
REPO_URL = 'https://github.com/R1ch1k/agent-reliability.git'

In [ ]:
# --- 0. confirm the hardware (decides feasibility) ---
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
print()
print('NEED ~80 GB: Qwen3.6-27B in BF16 is ~54 GB of weights; it does NOT fit a 40 GB A100.')
print('If memory.total is ~40 GB: Runtime > Change runtime type > A100 (80 GB), or fall back to a')
print('smaller CURRENT dense model at FULL precision. Do NOT 4-bit a 27B - quantization is the')
print('confound this white-box rung exists to avoid (near>=0.97 also guards against a broken quant).')


In [ ]:
# --- 1. install vLLM matched to THIS runtime's CUDA ---
# vLLM's PyPI wheel is built for ONE CUDA (currently 13); Colab ships torch for a different
# CUDA (e.g. cu128), so a plain `pip install vllm` leaves vllm._C unable to find libcudart
# -> ImportError: libcudart.so.XX. Install via uv with --torch-backend=auto + --reinstall so
# torch + vLLM + CUDA libs become ONE consistent stack matched to the GPU driver (~2-3 min).
!pip install -q uv huggingface_hub requests
!uv pip install --system --reinstall 'vllm>=0.19' --torch-backend=auto
!python -c "import vllm; print('vLLM', vllm.__version__); import vllm._C; print('vllm._C loads OK')"

In [ ]:
# --- 2. download weights (point local_dir at a mounted Drive path to persist across sessions) ---
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=MODEL_DIR)   # Apache-2.0, ungated (no token)
print('downloaded to', MODEL_DIR)


In [ ]:
# --- 3. serve at the FULL native window (no YaRN). Watch the log for the KV-cache capacity. ---
import subprocess, time, requests
args = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_DIR,
    '--served-model-name', SERVED,
    '--max-model-len', str(MAX_LEN),
    '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder',  # Qwen3.6 tool parser (card)
    '--enable-prefix-caching',          # reuse the shared manual prefix within a run
    '--gpu-memory-utilization', str(GMEM),
    '--port', '8000',
]
if KV_DTYPE != 'auto':
    args += ['--kv-cache-dtype', KV_DTYPE]   # fp8 halves KV memory to reach the full window (CONFOUND)
logf = open('/content/vllm.log', 'w')
srv = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for _ in range(360):                    # up to 30 min: 27B download-to-GPU load is slow
    try:
        if requests.get('http://localhost:8000/health').ok:
            ok = True; print('server UP  (KV dtype =', KV_DTYPE, '| max_model_len =', MAX_LEN, ')'); break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print('server did NOT come up - last log lines:')
    print(''.join(open('/content/vllm.log').readlines()[-40:]))
    print("If the log says the KV cache cannot hold max_model_len: either LOWER MAX_LEN, or set")
    print("KV_DTYPE='fp8' and SERVED='open-model-fp8kv' in the config cell, then re-run this cell.")

In [ ]:
# --- DEBUG: server health snapshot. Run anytime the kernel is free (after serve / between sweeps). ---
# Tells you whether ctx ~253k actually fits the KV cache, current GPU memory, and recent vLLM activity.
import subprocess
print('=== GPU memory (total, used, free) ===')
print(subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.used,memory.free',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
log = open('/content/vllm.log').read().splitlines()
print('\n=== vLLM KV-cache / capacity lines (how many tokens fit at once) ===')
for ln in log:
    if any(k in ln for k in ('KV cache', 'GPU KV', 'Maximum concurrency', 'GPU memory',
                             'token blocks', 'Available')):
        print(ln)
print('\n=== vLLM log tail (last 20 lines: recent requests / errors) ===')
print('\n'.join(log[-20:]))

In [ ]:
# --- 4. point the OpenAI SDK at the local server + force NON-thinking mode ---
import os, json
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'EMPTY'   # vLLM needs no auth
# Qwen3.6 THINKS BY DEFAULT and has no /no_think switch. The API panel was all non-reasoning,
# so for a capability-controlled comparison we force thinking OFF. The probe reads this env var
# (PROBE_OPENAI_EXTRA_BODY) and passes it as extra_body on every call.
os.environ['PROBE_OPENAI_EXTRA_BODY'] = json.dumps({'chat_template_kwargs': {'enable_thinking': False}})

from openai import OpenAI
r = OpenAI().chat.completions.create(
    model=SERVED,
    messages=[{'role': 'user', 'content': 'reply with the single word OK'}],
    max_tokens=16,
    extra_body=json.loads(os.environ['PROBE_OPENAI_EXTRA_BODY']),
)
print(repr(r.choices[0].message.content))   # expect 'OK' with NO think block


In [ ]:
# --- 5. clone harness + NEAR-only smoke at the top fills, --workers 1 --verbose (does it even run?) ---
!git clone -q $REPO_URL /content/probe
%cd /content/probe
# --workers 1 = no KV queue (the workers=5 queue is what timed out 172k). --verbose = live per-run lines.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near --fills $HI_FILLS --runs 3 --needles 3 --depth 0.5 --calib 0 --workers $WORKERS --verbose --timeout $TIMEOUT

## Go / no-go (push past 212k)

The smoke (cell 5) is the "does it even run at workers=1?" check. With `--verbose` you get a
**timestamped line per completed run**, so a hang is obvious (no new line for minutes) instead of a
silent stall, and `--timeout` makes a genuinely stuck call fail fast + retry rather than hang ~10 min.

- **near holds ~1.0 and each run completes in ~1-3 min** → the KV-queue timeout is gone; run cell 6.
- **a run stalls / times out even at workers=1** → the GPU genuinely can't serve ctx ~253k fast enough.
  Run the **DEBUG** cell (right after serve) to read KV-cache capacity + GPU memory + recent vLLM log;
  if 253k doesn't fit/serve, drop the top fill (e.g. `HI_FILLS = '160000'`) and re-run.

Cell 6 runs near+distance at the top fills and **appends** to the curve — combine with the banked
8k-212k data. **Record the confound:** Qwen3.6-27B, BF16, KV dtype auto, max-model-len 262144, YaRN OFF.

In [ ]:
# --- 6. PUSH sweep: near+distance at the top fills, --workers 1 --verbose --timeout (live + fail-fast). ---
# Live timestamped per-run lines; one request at a time so ctx ~253k never queues behind another.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills $HI_FILLS --runs 8 --needles 3 --depth 0.5 --calib 0 --workers $WORKERS --verbose --timeout $TIMEOUT

In [ ]:
# --- 7. DISENTANGLING pass — STILL DEFERRED (the curve is flat; nothing to disentangle). Skip unless distance bent. ---
# --padding inert + --fixed-needle-seed isolate raw context-LENGTH from search-space size.
!python reliability_probe_distance.py --provider openai --model $SERVED --conditions near distance --fills $HI_FILLS --runs 8 --needles 3 --depth 0.5 --padding inert --fixed-needle-seed 7 --workers $WORKERS --verbose --timeout $TIMEOUT

## Save + combine onto the ladder

Download (cell 8 below), then on your machine:
1. Put today's high-fill `dist_results_*.jsonl` next to yesterday's `colab_runs/dist_results_20260622_210615.jsonl` (8k-64k).
2. **If both are BF16-KV** (KV_DTYPE stayed 'auto'): one consistent `open-model` curve — add both files to `canonical_manifest.txt`, add `'open-model'` to the `order` list in `analyze_curves.py`, then re-run `analyze_curves.py` + `make_figures.py`.
3. **If today used fp8 KV** ('open-model-fp8kv'): keep it as a separate, confound-flagged high-fill arm. The near control + the BF16/fp8 behaviour at the overlap is your fp8-vs-BF16 cross-check.
4. The **inert-padded** disentangling file (if you ran cell 7) stays OUT of `canonical_manifest.txt`.

In [ ]:
# --- 8. zip + download all results (Colab is ephemeral) ---
from google.colab import files
import glob, zipfile
out = '/content/probe_results.zip'
with zipfile.ZipFile(out, 'w') as z:
    for f in glob.glob('/content/probe/dist_results_*.jsonl'):
        z.write(f, arcname=f.split('/')[-1])
    z.write('/content/vllm.log', arcname='vllm.log')
files.download(out)
print('zipped + downloading', out)
